# 04 · Limpieza de datos
Caso 1 · Reformas Alzina. A partir de las tablas `bruto_` se crean tablas `limpio_` con los
problemas del registro de limpieza resueltos. Las tablas `bruto_` no se modifican nunca, así que
este cuaderno se puede ejecutar entero (**Run All**) tantas veces como haga falta.

Los números de **ID** se refieren al registro de limpieza (`registro_limpieza.xlsx`).

| Tabla limpia | Qué contiene |
|---|---|
| `limpio_obras` | 223 obras, con tipo de trabajo unificado y cliente |
| `limpio_facturas` | 343 facturas de los dos programas, con su obra y su cliente |
| `limpio_clientes` | 166 clientes y su tipo (particular, empresa o comunidad) |
| `limpio_gastos` | Facturas de proveedores sin duplicados y con categorías unificadas |
| `limpio_partes` | Horas de los operarios por semana y obra |
| `limpio_nominas` | Nóminas con el coste para la empresa |
| `limpio_cobros` y `limpio_cobros_facturas` | Cobros del banco y la factura a la que corresponde cada uno |
| `estado_facturas` (vista) | Cuánto se ha cobrado de cada factura según el banco |

In [ ]:
import re
import sqlite3
import pandas as pd
from pathlib import Path

PROYECTO = Path(r"C:\Users\Annag\Desktop\portfolio\caso1_reformas_alzina_datos")
bd = sqlite3.connect(PROYECTO / "sql" / "alzina.db")

def sql(consulta):
    return pd.read_sql(consulta, bd)

## 1. Obras
- **ID 19** · Obra duplicada OB-25-055 → `SELECT DISTINCT` elimina las filas idénticas.
- **ID 7** · Tipo de trabajo escrito de 20 formas → `CASE` con la tabla de equivalencias, 6 tipos.
  La tabla va escrita a mano para que quede claro qué se agrupa con qué ("Reparación" va con
  "Mantenimiento" porque los dos son trabajos pequeños y cortos).
- **ID 20** · Fechas de inicio en dos formatos → todas a aaaa-mm-dd.
- **ID 8** · Presupuesto como texto con € → convertido a número.

In [ ]:
bd.executescript("""
DROP TABLE IF EXISTS limpio_obras;
CREATE TABLE limpio_obras AS
SELECT DISTINCT
    codigo,
    TRIM(cliente) AS cliente_original,
    poblacion,
    CASE
        WHEN tipo_trabajo IN ('Manten.', 'Mantenimiento', 'Reparación', 'reparacion') THEN 'Mantenimiento'
        WHEN tipo_trabajo IN ('PINTURA', 'Pintura', 'pintura') THEN 'Pintura'
        WHEN tipo_trabajo IN ('BAÑO completo', 'Baño', 'Reforma baño', 'baño') THEN 'Baño'
        WHEN tipo_trabajo IN ('Cocina', 'Reforma cocina', 'cocina') THEN 'Cocina'
        WHEN tipo_trabajo IN ('Integral', 'Reforma integral', 'Rehabilitación') THEN 'Reforma integral'
        WHEN tipo_trabajo IN ('Adecuación local', 'Local', 'Local comercial') THEN 'Local'
    END AS tipo,
    CASE
        WHEN fecha_inicio LIKE '__/__/____'
            THEN SUBSTR(fecha_inicio, 7, 4) || '-' || SUBSTR(fecha_inicio, 4, 2) || '-' || SUBSTR(fecha_inicio, 1, 2)
        ELSE SUBSTR(fecha_inicio, 1, 10)
    END AS fecha_inicio,
    SUBSTR(fecha_fin, 1, 10) AS fecha_fin,
    CASE
        WHEN presupuesto LIKE '%€%'
            THEN CAST(REPLACE(REPLACE(TRIM(REPLACE(presupuesto, '€', '')), '.', ''), ',', '.') AS REAL)
        ELSE CAST(presupuesto AS REAL)
    END AS presupuesto,
    observaciones
FROM bruto_obras;
""")
sql("SELECT * FROM limpio_obras LIMIT 5")

In [ ]:
# Comprobación: 223 obras, 223 códigos, 6 tipos, 0 fechas mal, 0 presupuestos vacíos
sql("""
SELECT COUNT(*) AS obras,
       COUNT(DISTINCT codigo) AS codigos_distintos,
       COUNT(DISTINCT tipo) AS tipos,
       SUM(fecha_inicio NOT LIKE '20__-__-__') AS fechas_mal,
       SUM(presupuesto IS NULL) AS presupuestos_vacios
FROM limpio_obras
""")

## 2. Facturas
Se juntan los dos ficheros en una sola tabla con `UNION ALL`.
- **ID 5** · 7 facturas en los dos ficheros → del fichero antiguo se excluyen las que ya están en el nuevo.
- **ID 6** · Formatos distintos en cada programa → mismas columnas, fechas aaaa-mm-dd, IVA como número.
- **ID 3** · Importes como texto en el fichero antiguo → convertidos a número.
- **ID 18** · Fechas con hora → solo la fecha.

Se guarda en qué programa estaba cada factura (`programa`) y el estado apuntado a mano
(`estado_apuntado`), que no se usará para saber qué está cobrado (**ID 16**).

In [ ]:
bd.executescript("""
DROP TABLE IF EXISTS limpio_facturas;
CREATE TABLE limpio_facturas AS
-- Facturas del programa antiguo (menos las 7 que se pasaron al programa nuevo)
SELECT
    no_factura AS num_factura,
    'antiguo' AS programa,
    SUBSTR(fecha, 7, 4) || '-' || SUBSTR(fecha, 4, 2) || '-' || SUBSTR(fecha, 1, 2) AS fecha,
    TRIM(cliente) AS cliente_original,
    NULLIF(TRIM(nif_cif), '') AS nif,
    concepto,
    CAST(REPLACE(REPLACE(base_imponible, '.', ''), ',', '.') AS REAL) AS base,
    CAST(pct_iva AS INTEGER) AS iva_pct,
    CAST(REPLACE(REPLACE(cuota_iva, '.', ''), ',', '.') AS REAL) AS cuota_iva,
    CAST(REPLACE(REPLACE(total, '.', ''), ',', '.') AS REAL) AS total,
    SUBSTR(vencimiento, 7, 4) || '-' || SUBSTR(vencimiento, 4, 2) || '-' || SUBSTR(vencimiento, 1, 2) AS vencimiento,
    NULLIF(TRIM(factura_rectificada), '') AS rectifica_a,
    NULL AS estado_apuntado
FROM bruto_facturas_antiguas
WHERE no_factura NOT IN (SELECT numero FROM bruto_facturas_nuevas)

UNION ALL

-- Facturas del programa nuevo (incluye las 7 migradas)
SELECT
    numero,
    'nuevo',
    SUBSTR(fecha_emision, 1, 10),
    TRIM(cliente),
    NULLIF(TRIM(cif_nif), ''),
    descripcion,
    CAST(base AS REAL),
    CAST(REPLACE(iva_pct, '%', '') AS INTEGER),
    CAST(importe_iva AS REAL),
    CAST(total AS REAL),
    SUBSTR(fecha_vencimiento, 1, 10),
    NULLIF(TRIM(rectifica_a), ''),
    estado
FROM bruto_facturas_nuevas;
""")
sql("SELECT * FROM limpio_facturas LIMIT 5")

In [ ]:
# Comprobación: 343 facturas (152 + 191), sin repetidas, sin fechas mal, sin importes vacíos
sql("""
SELECT COUNT(*) AS facturas,
       COUNT(DISTINCT num_factura) AS numeros_distintos,
       SUM(programa = 'antiguo') AS del_antiguo,
       SUM(programa = 'nuevo') AS del_nuevo,
       SUM(fecha NOT LIKE '20__-__-__') AS fechas_mal,
       SUM(base IS NULL OR total IS NULL) AS importes_vacios
FROM limpio_facturas
""")

### Total que no cuadra con base + IVA (ID 30)
Tres facturas del programa antiguo tienen un total redondo que no es igual a base + IVA.
En la sección 7 se ve en el banco que **los clientes pagaron base + IVA completo**, así que el
total redondeado es un error al apuntarlo, no un descuento. Se corrige: total = base + IVA.

In [ ]:
sql("""
SELECT num_factura, fecha, cliente_original, base, cuota_iva, total AS total_apuntado,
       ROUND(base + cuota_iva, 2) AS base_mas_iva
FROM limpio_facturas
WHERE ABS(total - (base + cuota_iva)) > 0.01
""")

In [ ]:
bd.execute("""
UPDATE limpio_facturas
SET total = ROUND(base + cuota_iva, 2)
WHERE ABS(total - (base + cuota_iva)) > 0.01
""")
bd.commit()

### Facturas rectificativas (ID 17)
Las 4 facturas con estado "Emitida" tienen importe negativo y la columna `rectifica_a` indica qué
factura corrigen: son abonos o descuentos. Se mantienen tal cual y restan de lo facturado.

In [ ]:
sql("""
SELECT num_factura, fecha, cliente_original, base, total, rectifica_a, estado_apuntado
FROM limpio_facturas
WHERE base < 0
""")

## 3. Clientes
- **ID 4** · El mismo cliente escrito de varias formas.
- **ID 15** · Facturas sin NIF.

Se hace en Python porque SQL no sabe transformar texto bien (no sabe pasar "LÓPEZ GIL" a "López Gil").
1. Normalizar el nombre: "LÓPEZ GIL, Pilar" → "Pilar López Gil". Las empresas no se tocan.
2. Si un nombre aparece alguna vez con NIF, todas sus facturas heredan ese NIF.
3. Cada cliente se identifica por su NIF; si no tiene, por su nombre normalizado.
4. Nombre oficial: el que no está todo en mayúsculas y más se repite.
5. Se crea `limpio_clientes` y se añade `cliente_id` a `limpio_facturas` y `limpio_obras`.

In [ ]:
import re

EMPRESA = re.compile(r"\b(S\.?L\.?P?|S\.?A\.?|SLP|SL|SA)\b|comunitat|comunidad|cdad", re.IGNORECASE)

def normalizar_nombre(nombre):
    """'LÓPEZ GIL, Pilar' -> 'Pilar López Gil'. Las empresas no se tocan."""
    nombre = " ".join(str(nombre).split())          # quita espacios dobles y al final
    if "," in nombre and not EMPRESA.search(nombre):
        apellidos, nombre_pila = [p.strip() for p in nombre.split(",", 1)]
        nombre = f"{nombre_pila} {apellidos.title()}"
    return nombre

# 1. Todas las apariciones de clientes, en facturas y en obras
fac = pd.read_sql("SELECT rowid AS fila, cliente_original, nif FROM limpio_facturas", bd)
obr = pd.read_sql("SELECT codigo, cliente_original FROM limpio_obras", bd)
fac["nombre"] = fac.cliente_original.map(normalizar_nombre)
obr["nombre"] = obr.cliente_original.map(normalizar_nombre)

# 2. Cada nombre que alguna vez aparece con NIF hereda ese NIF
nif_de_nombre = (fac.dropna(subset=["nif"]).groupby("nombre").nif
                 .agg(lambda x: x.mode().iloc[0]))
fac["nif"] = fac.nif.fillna(fac.nombre.map(nif_de_nombre))
obr["nif"] = obr.nombre.map(nif_de_nombre)

# 3. Identificador del cliente: el NIF si lo hay; si no, el nombre normalizado
fac["cliente_id"] = fac.nif.fillna(fac.nombre)
obr["cliente_id"] = obr.nif.fillna(obr.nombre)

# 4. Nombre oficial de cada cliente: el que no está todo en mayúsculas,
#    y entre esos, el que más se repite (y si empatan, el más largo)
def nombre_oficial(nombres):
    cuenta = nombres.value_counts()
    return max(cuenta.index, key=lambda n: (not n.isupper(), cuenta[n], len(n)))

todos = pd.concat([fac[["cliente_id", "nombre", "nif"]], obr[["cliente_id", "nombre", "nif"]]])
clientes = (todos.groupby("cliente_id")
            .agg(cliente=("nombre", nombre_oficial), nif=("nif", "first"))
            .reset_index())
clientes["tipo_cliente"] = clientes.cliente.map(
    lambda n: "Comunidad" if re.search(r"comunitat|comunidad", n, re.I)
    else "Empresa" if EMPRESA.search(n) else "Particular")

# 5. Guardar: tabla de clientes y columna cliente_id en facturas y obras
clientes.to_sql("limpio_clientes", bd, if_exists="replace", index=False)
for tabla, df, clave in [("limpio_facturas", fac, "fila"), ("limpio_obras", obr, "codigo")]:
    columnas = [c[1] for c in bd.execute(f"PRAGMA table_info({tabla})")]
    if "cliente_id" not in columnas:
        bd.execute(f"ALTER TABLE {tabla} ADD COLUMN cliente_id TEXT")
    campo = "rowid" if clave == "fila" else "codigo"
    bd.executemany(f"UPDATE {tabla} SET cliente_id = ? WHERE {campo} = ?",
                   df[["cliente_id", clave]].itertuples(index=False, name=None))
bd.commit()

In [ ]:
# Comprobación: 166 clientes y ninguna factura ni obra sin cliente
sql("""
SELECT (SELECT COUNT(*) FROM limpio_clientes) AS clientes,
       (SELECT COUNT(*) FROM limpio_clientes WHERE tipo_cliente = 'Particular') AS particulares,
       (SELECT COUNT(*) FROM limpio_clientes WHERE tipo_cliente <> 'Particular') AS empresas_y_comunidades,
       (SELECT SUM(cliente_id IS NULL) FROM limpio_facturas) AS facturas_sin_cliente,
       (SELECT SUM(cliente_id IS NULL) FROM limpio_obras) AS obras_sin_cliente
""")

In [ ]:
# Los 10 clientes que más han facturado
sql("""
SELECT c.cliente, c.tipo_cliente, COUNT(*) AS facturas, ROUND(SUM(f.base)) AS facturado_sin_iva
FROM limpio_facturas AS f
JOIN limpio_clientes AS c USING (cliente_id)
GROUP BY c.cliente_id
ORDER BY facturado_sin_iva DESC
LIMIT 10
""")

## 4. Gastos
- **ID 21** · 5 facturas registradas dos veces → `SELECT DISTINCT`.
- **ID 22** · Categorías escritas de 16 formas → se unifican Materiales y Subcontratas.
- **ID 11** · Proveedores escritos de varias formas → nombre oficial por NIF, con `ROW_NUMBER()`.
- **ID 23** · Códigos de obra sin "OB-" → se añade.

El **ID 10** (307 gastos de materiales sin obra) no se puede resolver con los datos: no hay
nada que diga a qué obra van. Se decidirá en el análisis cómo repartirlos.

In [ ]:
bd.executescript("""
DROP TABLE IF EXISTS limpio_gastos;
CREATE TABLE limpio_gastos AS
WITH sin_duplicados AS (
    SELECT DISTINCT * FROM bruto_gastos
),
nombre_oficial AS (
    -- Por cada NIF, el nombre que no está todo en mayúsculas y que más se repite
    SELECT nif_proveedor, proveedor AS proveedor_oficial
    FROM (
        SELECT nif_proveedor, proveedor,
               ROW_NUMBER() OVER (
                   PARTITION BY nif_proveedor
                   ORDER BY (UPPER(proveedor) <> proveedor) DESC, COUNT(*) DESC, LENGTH(proveedor) DESC
               ) AS orden
        FROM bruto_gastos
        GROUP BY nif_proveedor, proveedor
    )
    WHERE orden = 1
)
SELECT
    SUBSTR(g.fecha_factura, 1, 10) AS fecha,
    g.no_factura AS num_factura,
    n.proveedor_oficial AS proveedor,
    g.nif_proveedor,
    CASE
        WHEN g.categoria IN ('Compras material', 'Material', 'Materiales', 'Materiales obra') THEN 'Materiales'
        WHEN g.categoria IN ('Subcontrata', 'Subcontratas', 'Trabajos otras empresas') THEN 'Subcontratas'
        ELSE g.categoria
    END AS categoria,
    CASE
        WHEN g.obra IS NULL OR g.obra = '' THEN NULL
        WHEN g.obra LIKE 'OB-%' THEN g.obra
        ELSE 'OB-' || g.obra
    END AS obra,
    CAST(g.base AS REAL) AS base,
    CAST(g.pct_iva AS INTEGER) AS iva_pct,
    CAST(g.cuota_iva AS REAL) AS cuota_iva,
    CAST(g.retencion_irpf AS REAL) AS retencion_irpf,
    CAST(g.total_a_pagar AS REAL) AS total_a_pagar
FROM sin_duplicados AS g
JOIN nombre_oficial AS n USING (nif_proveedor);
""")
sql("SELECT * FROM limpio_gastos LIMIT 5")

In [ ]:
# Comprobación: 1.229 gastos, 19 proveedores = 19 NIF, 11 categorías, 0 obras inexistentes
sql("""
SELECT COUNT(*) AS gastos,
       COUNT(DISTINCT proveedor) AS proveedores,
       COUNT(DISTINCT nif_proveedor) AS nifs,
       COUNT(DISTINCT categoria) AS categorias,
       SUM(obra IS NOT NULL AND obra NOT IN (SELECT codigo FROM limpio_obras)) AS obras_inexistentes,
       SUM(categoria = 'Materiales' AND obra IS NULL) AS materiales_sin_obra
FROM limpio_gastos
""")

## 5. Partes de horas y nóminas
- **ID 14** · 4 operarios escritos de 10 formas → nombre completo de las nóminas.
- **ID 25** · 85 y 80 horas en una semana → se dividen entre 10 (la coma se perdió al teclear).
- **ID 26** · Partes con 0 horas → se eliminan.
- **ID 27** · Códigos de obra mal escritos → se añade el guion que falta y los ceros.

Las nóminas no tienen problemas: se convierten los importes a número y se añade el coste para
la empresa (bruto + Seguridad Social a cargo de la empresa).

In [ ]:
bd.executescript("""
DROP TABLE IF EXISTS limpio_partes;
CREATE TABLE limpio_partes AS
WITH paso1 AS (
    SELECT
        semana_inicio,
        CASE
            WHEN operario IN ('Abdel', 'A. Benali', 'Abdel Benali') THEN 'Abdel Benali'
            WHEN operario IN ('Iván', 'Ivan Moreno', 'Iván Moreno') THEN 'Iván Moreno'
            WHEN operario IN ('Marc', 'Marc Vidal') THEN 'Marc Vidal'
            WHEN operario IN ('Ruben Ortega', 'Rubén Ortega') THEN 'Rubén Ortega'
        END AS operario,
        -- OB24-005 -> OB-24-005 (falta el primer guion)
        CASE WHEN obra LIKE 'OB__-%' THEN 'OB-' || SUBSTR(obra, 3) ELSE obra END AS obra,
        CAST(horas AS REAL) AS horas
    FROM bruto_partes
)
SELECT
    semana_inicio,
    operario,
    -- OB-24-15 -> OB-24-015 (faltan los ceros a la izquierda)
    SUBSTR(obra, 1, 6) || SUBSTR('000' || SUBSTR(obra, 7), -3) AS obra,
    -- 85 y 80 horas en una semana: la coma se perdió al teclear
    CASE WHEN horas > 50 THEN horas / 10 ELSE horas END AS horas
FROM paso1
WHERE horas > 0;

DROP TABLE IF EXISTS limpio_nominas;
CREATE TABLE limpio_nominas AS
SELECT
    mes,
    trabajador,
    categoria,
    concepto,
    CAST(devengado AS REAL) AS devengado,
    CAST(ss_empresa AS REAL) AS ss_empresa,
    CAST(ss_trabajador AS REAL) AS ss_trabajador,
    CAST(irpf AS REAL) AS irpf,
    CAST(liquido AS REAL) AS liquido,
    CAST(devengado AS REAL) + CAST(ss_empresa AS REAL) AS coste_empresa
FROM bruto_nominas;
""")
sql("SELECT * FROM limpio_partes LIMIT 5")

In [ ]:
# Comprobación: 1.748 partes, 4 operarios, máximo razonable, 0 obras inexistentes, 228 nóminas
sql("""
SELECT COUNT(*) AS partes,
       COUNT(DISTINCT operario) AS operarios,
       SUM(operario IS NULL) AS operarios_sin_nombre,
       MIN(horas) AS minimo_horas,
       MAX(horas) AS maximo_horas,
       SUM(obra NOT IN (SELECT codigo FROM limpio_obras)) AS obras_inexistentes,
       (SELECT COUNT(*) FROM limpio_nominas) AS nominas
FROM limpio_partes
""")

## 6. Obra de cada factura y fecha de fin de las obras
Para calcular el margen de cada obra, cada factura tiene que saber a qué obra pertenece.
1. **Código en el concepto**, escrito de cualquiera de sus formas (OB-24-031, OB24-031, ob-24-31).
   Se busca con una expresión regular.
2. **Rectificativas sin código** → la obra de la factura que corrigen.
3. **El resto** → la obra del mismo cliente cuyas fechas encajan con la de la factura.

Se guarda con qué método se encontró cada una (`metodo_obra`), para saber cuáles son seguras.

Después se resuelve el **ID 9**: las 4 obras terminadas sin fecha de fin reciben la fecha de su
factura final. Las 3 obras "En curso" se dejan sin fecha, que es lo correcto.

In [ ]:
import re

fac = pd.read_sql("SELECT rowid AS fila, num_factura, fecha, cliente_id, concepto, rectifica_a FROM limpio_facturas", bd)
obras = pd.read_sql("SELECT codigo, cliente_id, fecha_inicio, fecha_fin FROM limpio_obras", bd)

# 1. Código escrito en el concepto, en cualquiera de sus formas: OB-24-031, OB24-031, ob-24-31
patron = re.compile(r"ob-?(\d{2})-(\d{1,3})", re.IGNORECASE)
def codigo_del_concepto(texto):
    m = patron.search(texto or "")
    return f"OB-{m.group(1)}-{int(m.group(2)):03d}" if m else None

fac["obra"] = fac.concepto.map(codigo_del_concepto)
fac["metodo_obra"] = fac.obra.map(lambda x: "concepto" if x else None)

# 2. Rectificativas sin código: la misma obra que la factura que corrigen
obra_de_factura = fac.set_index("num_factura").obra
sin = fac.obra.isna() & fac.rectifica_a.notna()
fac.loc[sin, "obra"] = fac.loc[sin, "rectifica_a"].map(obra_de_factura)
fac.loc[sin & fac.obra.notna(), "metodo_obra"] = "factura rectificada"

# 3. El resto: la obra del mismo cliente cuyas fechas encajan con la fecha de la factura
#    (desde 7 días antes del inicio hasta 45 días después del fin; si sigue abierta, sin límite).
#    Si encajan varias, la que empezó más cerca de la fecha de la factura.
obras["ini"] = pd.to_datetime(obras.fecha_inicio) - pd.Timedelta(days=7)
obras["fin"] = pd.to_datetime(obras.fecha_fin).fillna(pd.Timestamp("2099-12-31")) + pd.Timedelta(days=45)
def obra_por_fechas(f):
    fecha = pd.Timestamp(f.fecha)
    c = obras[(obras.cliente_id == f.cliente_id) & (obras.ini <= fecha) & (fecha <= obras.fin)]
    if c.empty:
        return None
    return c.loc[(pd.to_datetime(c.fecha_inicio) - fecha).abs().idxmin(), "codigo"]

sin = fac.obra.isna()
fac.loc[sin, "obra"] = fac[sin].apply(obra_por_fechas, axis=1)
fac.loc[sin & fac.obra.notna(), "metodo_obra"] = "cliente y fechas"

# 4. Guardar en limpio_facturas
for col in ["obra", "metodo_obra"]:
    if col not in [c[1] for c in bd.execute("PRAGMA table_info(limpio_facturas)")]:
        bd.execute(f"ALTER TABLE limpio_facturas ADD COLUMN {col} TEXT")
bd.executemany("UPDATE limpio_facturas SET obra = ?, metodo_obra = ? WHERE rowid = ?",
               fac[["obra", "metodo_obra", "fila"]].itertuples(index=False, name=None))
bd.commit()

In [ ]:
bd.executescript("""
-- Obras terminadas sin fecha de fin: la fecha de su factura final.
-- Las obras "En curso" se dejan sin fecha.
UPDATE limpio_obras
SET fecha_fin = (
        SELECT MAX(f.fecha)
        FROM limpio_facturas AS f
        WHERE f.obra = limpio_obras.codigo
          AND (f.concepto LIKE 'Final%' OR f.concepto LIKE 'Liquidación final%')
    )
WHERE fecha_fin IS NULL
  AND COALESCE(observaciones, '') <> 'En curso';
""")

In [ ]:
# Comprobación: todas las facturas tienen obra, todas existen y todas son del mismo cliente que su obra
sql("""
SELECT f.metodo_obra,
       COUNT(*) AS facturas,
       SUM(o.codigo IS NULL) AS obra_inexistente,
       SUM(o.cliente_id <> f.cliente_id) AS cliente_distinto
FROM limpio_facturas AS f
LEFT JOIN limpio_obras AS o ON o.codigo = f.obra
GROUP BY f.metodo_obra
""")

In [ ]:
# Las 4 obras que no tenían fecha de fin, y las 3 en curso
sql("""
SELECT codigo, tipo, fecha_inicio, fecha_fin, observaciones
FROM limpio_obras
WHERE codigo IN ('OB-24-001', 'OB-24-024', 'OB-24-080', 'OB-25-001')
   OR observaciones = 'En curso'
""")

## 7. Conciliación bancaria: qué factura paga cada cobro
Es el paso más importante: la columna Estado de las facturas no es fiable (**ID 16**) y 158 cobros
no dicen qué factura pagan (**ID 29**). El banco es la única fuente de verdad sobre el dinero.

**Método**
1. **Cliente de cada cobro:** el banco escribe el nombre en mayúsculas y cortado, así que se
   compara el principio del concepto con todas las formas en que aparece cada cliente.
2. **Primera pasada:** los cobros que citan número de factura se aplican a esas facturas.
3. **Segunda pasada**, del cobro más antiguo al más reciente, los que no citan ninguna:
   a) una factura abierta del cliente con exactamente ese importe;
   b) si no, una combinación de 2 a 4 facturas que sume exactamente ese importe
      (Habitat Mar y Finques Garbí pagan varias facturas en una transferencia);
   c) si no, las facturas abiertas más antiguas del cliente (pagos a plazos de la comunidad).
4. Las rectificativas restan de la factura que corrigen: el cliente paga el importe ya descontado.

La vista del banco se vuelve a crear aquí porque es temporal.

In [ ]:
bd.execute("DROP VIEW IF EXISTS banco")
bd.execute("""
CREATE TEMP VIEW banco AS
SELECT rowid AS fila,
       SUBSTR(fecha_operacion, 7, 4) || '-' || SUBSTR(fecha_operacion, 4, 2) || '-' || SUBSTR(fecha_operacion, 1, 2) AS fecha,
       concepto,
       CAST(REPLACE(REPLACE(importe, '.', ''), ',', '.') AS REAL) AS importe,
       CAST(REPLACE(REPLACE(saldo, '.', ''), ',', '.') AS REAL) AS saldo
FROM bruto_banco
""")

In [ ]:
import re
from collections import defaultdict

def clave(texto):
    """Mayúsculas, sin puntos ni espacios dobles: como escribe el banco los nombres."""
    return " ".join(str(texto).upper().replace(".", "").split())

# --- 1. Cobros del banco (entradas de dinero de clientes) ---------------------------
banco = pd.read_sql("""
    SELECT fila, fecha, concepto, importe FROM banco
    WHERE concepto LIKE 'TRANSF DE %' OR concepto LIKE 'BIZUM DE %'
    ORDER BY fila DESC
""", bd)
banco["texto"] = banco.concepto.str.replace(r"^(TRANSF DE|BIZUM DE) ", "", regex=True).map(clave)

# --- 2. ¿De qué cliente es cada cobro? ------------------------------------------------
# El banco escribe el nombre en mayúsculas y cortado (a unos 22 caracteres), así que se
# compara el principio del texto con todas las formas en que aparece cada cliente.
formas = pd.concat([
    pd.read_sql("SELECT cliente_original AS forma, cliente_id FROM limpio_facturas", bd),
    pd.read_sql("SELECT cliente_original AS forma, cliente_id FROM limpio_obras", bd),
    pd.read_sql("SELECT cliente AS forma, cliente_id FROM limpio_clientes", bd),
]).drop_duplicates()
formas["inicio"] = formas.forma.map(lambda f: clave(normalizar_nombre(f))[:20])
formas = pd.concat([formas, formas.assign(inicio=formas.forma.map(lambda f: clave(f)[:20]))]).drop_duplicates()

def cliente_del_cobro(texto):
    posibles = formas[formas.inicio.map(lambda i: texto.startswith(i))]
    if posibles.empty:
        return None
    return posibles.loc[posibles.inicio.str.len().idxmax(), "cliente_id"]

banco["cliente_id"] = banco.texto.map(cliente_del_cobro)

# --- 3. Números de factura que aparecen en el concepto --------------------------------
patron_factura = re.compile(r"\b(\d{4}/\d{4}|F\d{2}-\d{4})\b")
banco["facturas_citadas"] = banco.concepto.map(patron_factura.findall)

# --- 4. Importe pendiente de cada factura (las rectificativas restan de la que corrigen) ---
fac = pd.read_sql("SELECT num_factura, fecha, cliente_id, total, rectifica_a FROM limpio_facturas ORDER BY fecha, num_factura", bd)
pendiente = fac[fac.rectifica_a.isna()].set_index("num_factura").total.to_dict()
for r in fac[fac.rectifica_a.notna()].itertuples():
    pendiente[r.rectifica_a] = round(pendiente[r.rectifica_a] + r.total, 2)
cliente_de = fac.set_index("num_factura").cliente_id.to_dict()
fecha_de = fac.set_index("num_factura").fecha.to_dict()
abiertas = defaultdict(list)                                   # facturas por cliente, en orden de fecha
for n in fac[fac.rectifica_a.isna()].num_factura:
    abiertas[cliente_de[n]].append(n)

# --- 5. Emparejar cada cobro con sus facturas -----------------------------------------
#   Primera pasada: los cobros que citan número de factura (son seguros).
#   Segunda pasada, del más antiguo al más reciente, los que no citan ninguna:
#     a) una factura abierta del cliente con exactamente ese importe;
#     b) si no, una combinación de 2 a 4 facturas abiertas que sume exactamente ese importe;
#     c) si no, las facturas abiertas más antiguas del cliente, hasta agotar el cobro.
from itertools import combinations
aplicaciones = []
def aplicar(fila, factura, importe, metodo):
    aplicaciones.append((fila, factura, round(importe, 2), metodo))
    pendiente[factura] = round(pendiente[factura] - importe, 2)

banco["citadas"] = banco.facturas_citadas.map(lambda l: [n for n in l if n in pendiente])
for c in banco[banco.citadas.str.len() > 0].itertuples():
    resto = c.importe
    for n in c.citadas:
        parte = min(resto, max(pendiente[n], 0))
        if parte > 0:
            aplicar(c.fila, n, parte, "número de factura")
            resto -= parte
    if resto > 0.02:
        aplicaciones.append((c.fila, None, round(resto, 2), "sin factura"))

for c in banco[banco.citadas.str.len() == 0].itertuples():
    resto = c.importe
    abiertas_cliente = [n for n in abiertas.get(c.cliente_id, [])
                        if pendiente[n] > 0.005 and fecha_de[n] <= c.fecha]
    exacta = [n for n in abiertas_cliente if abs(pendiente[n] - resto) < 0.02]
    combinacion = None
    if not exacta:
        for k in (2, 3, 4):
            combinacion = next((cb for cb in combinations(abiertas_cliente[:12], k)
                                if abs(sum(pendiente[n] for n in cb) - resto) < 0.02), None)
            if combinacion:
                break
    if exacta:
        aplicar(c.fila, exacta[0], resto, "importe exacto"); resto = 0
    elif combinacion:
        for n in combinacion:
            parte = pendiente[n]
            aplicar(c.fila, n, parte, "suma de facturas"); resto -= parte
    else:
        for n in abiertas_cliente:
            if resto <= 0.005:
                break
            parte = min(resto, pendiente[n])
            aplicar(c.fila, n, parte, "facturas más antiguas"); resto -= parte
    if resto > 0.02:
        aplicaciones.append((c.fila, None, round(resto, 2), "sin factura"))

# --- 6. Guardar ---------------------------------------------------------------------
limpio_cobros = banco[["fila", "fecha", "concepto", "importe", "cliente_id"]].rename(columns={"fila": "fila_banco"})
limpio_cobros.to_sql("limpio_cobros", bd, if_exists="replace", index=False)
pd.DataFrame(aplicaciones, columns=["fila_banco", "num_factura", "importe", "metodo"]) \
  .to_sql("limpio_cobros_facturas", bd, if_exists="replace", index=False)
bd.commit()

In [ ]:
bd.executescript("""
DROP VIEW IF EXISTS estado_facturas;
CREATE VIEW estado_facturas AS
SELECT
    f.num_factura,
    f.fecha,
    f.vencimiento,
    f.cliente_id,
    f.obra,
    f.total + COALESCE((SELECT SUM(r.total) FROM limpio_facturas AS r WHERE r.rectifica_a = f.num_factura), 0) AS total_a_cobrar,
    COALESCE(SUM(cf.importe), 0) AS cobrado,
    MAX(c.fecha) AS fecha_ultimo_cobro,
    f.estado_apuntado
FROM limpio_facturas AS f
LEFT JOIN limpio_cobros_facturas AS cf ON cf.num_factura = f.num_factura
LEFT JOIN limpio_cobros AS c ON c.fila_banco = cf.fila_banco
WHERE f.rectifica_a IS NULL
GROUP BY f.num_factura;
""")

In [ ]:
# Comprobación: todos los cobros tienen cliente y casi todo el dinero encuentra su factura
sql("""
SELECT metodo, COUNT(*) AS aplicaciones, ROUND(SUM(importe), 2) AS importe
FROM limpio_cobros_facturas
GROUP BY metodo
ORDER BY importe DESC
""")

In [ ]:
sql("""
SELECT (SELECT COUNT(*) FROM limpio_cobros) AS cobros,
       (SELECT SUM(cliente_id IS NULL) FROM limpio_cobros) AS cobros_sin_cliente,
       ROUND(SUM(total_a_cobrar), 2) AS facturado_con_iva,
       ROUND(SUM(cobrado), 2) AS cobrado,
       ROUND(SUM(total_a_cobrar - cobrado), 2) AS pendiente_de_cobro,
       SUM(cobrado > total_a_cobrar + 0.02) AS facturas_cobradas_de_mas
FROM estado_facturas
""")

In [ ]:
# Facturas pendientes de cobro a 30/09/2026
sql("""
SELECT e.num_factura, e.fecha, e.vencimiento, c.cliente,
       ROUND(e.total_a_cobrar, 2) AS total, ROUND(e.cobrado, 2) AS cobrado,
       ROUND(e.total_a_cobrar - e.cobrado, 2) AS pendiente,
       CAST(JULIANDAY('2026-09-30') - JULIANDAY(e.vencimiento) AS INTEGER) AS dias_vencida,
       e.estado_apuntado
FROM estado_facturas AS e
JOIN limpio_clientes AS c USING (cliente_id)
WHERE e.total_a_cobrar - e.cobrado > 0.02
ORDER BY e.fecha
""")

In [ ]:
# ¿Es fiable el Estado apuntado a mano? (ID 16)
sql("""
SELECT estado_apuntado,
       SUM(total_a_cobrar - cobrado <= 0.02) AS cobradas_segun_banco,
       SUM(total_a_cobrar - cobrado > 0.02) AS pendientes_segun_banco
FROM estado_facturas
WHERE estado_apuntado IS NOT NULL
GROUP BY estado_apuntado
""")

## Mis observaciones · Limpieza

**Resultado**
- Las 7 tablas en bruto se han convertido en tablas limpias y conectadas entre sí por obra,
  cliente y factura. Todas las comprobaciones dan 0 errores.
- Facturas: 343 (152 del programa antiguo y 191 del nuevo). Facturado sin IVA: 523.481 € en 2024,
  629.009 € en 2025 y 513.735 € de enero a septiembre de 2026.
- Clientes: 198 nombres distintos se han unificado en 166 clientes (158 particulares, 7 empresas
  y 1 comunidad).
- Cada factura tiene su obra: 303 por el código del concepto y 40 por cliente y fechas.

**Correcciones que han cambiado lo que pensaba al explorar**
- **Totales redondeados (ID 30):** en la exploración supuse que eran un descuento. El banco
  demuestra que los clientes pagaron base + IVA completo, así que eran un error al apuntarlos.
  Se ha corregido el total.
- **Estado de las facturas (ID 16):** no es fiable. **4 facturas marcadas "Cobrada" no se han
  cobrado nunca** (F25-0008, F26-0053, F26-0060 y F26-0074: 45.738 € que Jordi cree que ya tiene),
  y 9 marcadas "Pendiente" o "Vencida" ya están cobradas.

**Hallazgos para el análisis**
- **Pendiente de cobro a 30/09/2026: 114.947 €** en 13 facturas.
- **3 facturas parecen impagadas** (llevan meses vencidas y sin ningún cobro): Carles Rubio Serra
  (11.362 €, mayo de 2025), Xavier Ruiz Font (8.950 €, octubre de 2025) y Restaurant Can Mas
  (554 €, septiembre de 2025).
- **Habitat Mar Inversiones** acumula 7 facturas sin cobrar (72.111 €), algunas vencidas desde
  el verano: paga cada vez más tarde.
- Las facturas de Mercè Roca Gil y Javier Torres Grau son de los últimos días de septiembre:
  es normal que aún no estén cobradas.

**Pendiente**
- **ID 10:** 307 gastos de materiales sin obra. Se decidirá cómo repartirlos al calcular márgenes.